In [1]:
import os

# 1. Define the path to our good Java
java21_path = "/usr/lib/jvm/java-21-openjdk-amd64"

# 2. Set JAVA_HOME
os.environ["JAVA_HOME"] = java21_path

# 3. The Magic: Shove Java 21 to the absolute front of the PATH
os.environ["PATH"] = f"{java21_path}/bin:" + os.environ.get("PATH", "")

In [3]:
import pyspark
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/16 20:18:49 WARN Utils: Your hostname, codespaces-472bba, resolves to a loopback address: 127.0.0.1; using 10.0.3.105 instead (on interface eth0)
26/03/16 20:18:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 20:18:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark.version

'4.1.1'

In [6]:
df = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/yellow/2025')

In [7]:
df = df.repartition(4)

In [8]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [9]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [10]:
df.select("tpep_pickup_datetime").show(5, truncate=False)

+--------------------+
|tpep_pickup_datetime|
+--------------------+
|2025-05-26 16:51:31 |
|2025-05-28 11:29:39 |
|2025-05-15 13:47:30 |
|2025-05-16 21:37:17 |
|2025-05-27 12:33:50 |
+--------------------+
only showing top 5 rows


In [15]:
df.createOrReplaceTempView("trips")

In [18]:
df_result = spark.sql("""
SELECT 
    COUNT (*)
FROM
    trips
WHERE
    tpep_pickup_datetime >= '2025-11-15 00:00:00'
    AND
    tpep_pickup_datetime <= '2025-11-16 00:00:00'
""").show()

+--------+
|count(1)|
+--------+
|  162608|
+--------+



In [33]:
df_result = spark.sql("""
WITH trip_times AS (
    SELECT
        unix_timestamp(tpep_pickup_datetime) AS put,
        unix_timestamp(tpep_dropoff_datetime) AS dot
    FROM
        trips
)
SELECT
    ROUND(MAX(dot - put) / 3600, 2) AS max_trip_time_hrs
FROM
    trip_times
WHERE
    dot > put
""").show()

+-----------------+
|max_trip_time_hrs|
+-----------------+
|           248.01|
+-----------------+



In [ ]:
# Q6: Download Taxi Zones CSV
# !wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-16 21:12:06--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.33.98, 13.35.33.60, 13.35.33.10, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.33.98|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-16 21:12:07 (1.16 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [ ]:
# Q6: Read & Write Taxi Zones CSV

spark \
    .read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv") \
    .write \
    .parquet("zones", mode='overwrite')

In [ ]:
# Q6: Read Zones PARQUET
df_zones = spark.read.parquet("zones")

In [51]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [45]:
# Q6: Yellow Taxi 2025 November data
df_y202511 = spark.read \
    .parquet("data/pq/yellow/2025/11")

In [47]:
df_y202511.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [48]:
df_join = df_y202511.join(df_zones, df_y202511.PULocationID == df_zones.LocationID)

In [50]:
df_join.createOrReplaceTempView("trips_w_zones")

In [53]:
df_result = spark.sql("""
    SELECT
        Zone,
        COUNT(1) AS zfreq
    FROM
        trips_w_zones
    GROUP BY
        Zone
    ORDER BY
        zfreq ASC
""").show()

+--------------------+-----+
|                Zone|zfreq|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
|   Rossville/Woodrow|    4|
|         Great Kills|    4|
| Green-Wood Cemetery|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|New Dorp/Midland ...|   14|
|             Oakwood|   14|
|        Crotona Park|   14|
|       West Brighton|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows


In [ ]:
#spark.stop()